# # Amazon Product Data Cleaning and Preprocessing
 
This notebook cleans and prepares the scraped Amazon product data for analysis.
It handles duplicates, missing values, invalid data, and performs necessary transformations.

## 1. Import Required Libraries

In [13]:
import pandas as pd
import numpy as np
import json
import re
import os
import glob
from datetime import datetime
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 1000)

print("Libraries imported successfully")

Libraries imported successfully


## Load Data from CSV Files in Output Folder

In [ ]:
def find_csv_files(output_folder='output'):
    """Find CSV files in the output folder"""
    csv_files = {}
    
    if not os.path.exists(output_folder):
        print(f"Output folder '{output_folder}' not found!")
        return csv_files
    
    # Find all CSV files
    all_csv_files = glob.glob(os.path.join(output_folder, '*.csv'))
    
    for file_path in all_csv_files:
        filename = os.path.basename(file_path)
        
        # Categorize files
        if 'product' in filename.lower():
            csv_files['products'] = file_path
        elif 'review' in filename.lower():
            csv_files['reviews'] = file_path
        elif 'keyword' in filename.lower():
            csv_files['keywords'] = file_path
    
    print(f"Found CSV files in '{output_folder}':")
    for key, path in csv_files.items():
        print(f"  - {key}: {os.path.basename(path)}")
    
    return csv_files

def load_products_from_csv(file_path):
    """Load products from CSV file"""
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
        print(f"Loaded {len(df)} products from {os.path.basename(file_path)}")
        print(f"Columns: {list(df.columns)}")
        return df
    except Exception as e:
        print(f"Error loading products CSV: {e}")
        return pd.DataFrame()

def load_reviews_from_products(products_df):
    import json
    import pandas as pd

    reviews = []

    for _, row in products_df.iterrows():
        value = row.get("reviews")

        if pd.isna(value) or not value:
            continue

        try:
            product_reviews = json.loads(value)

            for review in product_reviews:
                review["product_id"] = row["product_id"]
                review["asin"] = row["asin"]

            reviews.extend(product_reviews)

        except (json.JSONDecodeError, TypeError) as e:
            print(
                f"Could not parse reviews for "
                f"product_id={row.get('product_id')}, "
                f"asin={row.get('asin')}: {e}"
            )

    return pd.DataFrame(reviews)

In [ ]:
csv_files = find_csv_files('../output')

# Load data from CSV files
products_df = pd.DataFrame()
reviews_df = pd.DataFrame()


if 'products' in csv_files:
    products_df = load_products_from_csv(csv_files['products'])
    reviews_df = load_reviews_from_products(products_df)

    print(f"Reviews DataFrame shape: {reviews_df.shape}")
print(f"\nFinal data loaded:")
print(f"  Products: {len(products_df)}")
print(f"  Reviews: {len(reviews_df)}")

Found CSV files in '../output':
  - products: amazon_products_with_reviews_20260830_235401.csv
Loaded 621 products from amazon_products_with_reviews_20260830_235401.csv
Columns: ['product_id', 'asin', 'title', 'full_title', 'url', 'image_url', 'price', 'current_price', 'original_price', 'discount', 'rating', 'detailed_rating', 'reviews_count', 'detailed_reviews_count', 'brand', 'manufacturer', 'availability', 'description', 'features', 'dimensions', 'best_sellers_rank', 'date_first_available', 'is_prime', 'is_sponsored', 'video_url', 'video_thumbnail', 'keyword', 'scraped_at', 'created_at', 'updated_at', 'technical_details', 'reviews', 'reviews_count_actual', 'tech_brand_name', 'tech_hand_orientation', 'tech_upc', 'tech_set_name', 'tech_number_of_items', 'tech_included_components', 'tech_instrument', 'tech_item_weight', 'tech_model_name', 'tech_model_number', 'tech_manufacturer_part_number', 'tech_warranty_description', 'tech_color', 'tech_top_material_type', 'tech_body_material_type',

## Initial Data Exploration

In [16]:
if not products_df.empty:
    # Display first few rows
    print("=== Products Data ===")
    print(products_df.head())
    print(f"\nProducts shape: {products_df.shape}")
    
    # Display data types
    print("\n=== Data Types ===")
    print(products_df.dtypes)
    
    # Display missing values
    print("\n=== Missing Values ===")
    missing_values = products_df.isnull().sum()
    missing_percentage = (missing_values / len(products_df)) * 100
    
    missing_df = pd.DataFrame({
        'Missing Values': missing_values,
        'Percentage': missing_percentage
    })
    print(missing_df[missing_df['Missing Values'] > 0])
    
    # Basic statistics
    print("\n=== Basic Statistics ===")
    print(products_df.describe())
else:
    print("No products data loaded!")

# %%
if not reviews_df.empty:
    print("=== Reviews Data ===")
    print(reviews_df.head())
    print(f"\nReviews shape: {reviews_df.shape}")
    
    print("\n=== Missing Values in Reviews ===")
    missing_reviews = reviews_df.isnull().sum()
    print(missing_reviews[missing_reviews > 0])
else:
    print("No reviews data loaded!")

=== Products Data ===
   product_id        asin   title                                                                                           full_title                                                                                                  url                                                       image_url    price current_price original_price discount              rating         detailed_rating reviews_count detailed_reviews_count                   brand  manufacturer                        availability                                                                                          description                                                                                             features  dimensions                                                                                    best_sellers_rank date_first_available is_prime is_sponsored                                                         video_url                                             video_th

Data Cleaning Functions

In [17]:
def clean_price(price_str):
    """Clean and convert price string to float"""
    if pd.isna(price_str) or price_str == 'N/A' or price_str == '' or price_str is None:
        return np.nan
    
    if isinstance(price_str, (int, float)):
        return float(price_str)
    
    # Remove currency symbols and commas
    price_str = str(price_str).strip()
    
    # Extract price using regex
    price_match = re.search(r'[\d,]+\.?\d*', price_str)
    if price_match:
        price_str = price_match.group(0)
        price_str = price_str.replace(',', '')
        try:
            return float(price_str)
        except:
            return np.nan
    return np.nan

def clean_rating(rating_str):
    """Clean and convert rating string to float"""
    if pd.isna(rating_str) or rating_str == 'N/A' or rating_str == '' or rating_str is None:
        return np.nan
    
    if isinstance(rating_str, (int, float)):
        return float(rating_str)
    
    # Extract rating using regex
    rating_match = re.search(r'(\d+\.?\d*)', str(rating_str))
    if rating_match:
        try:
            return float(rating_match.group(1))
        except:
            return np.nan
    return np.nan

def clean_reviews_count(count_str):
    """Clean and convert reviews count to integer"""
    if pd.isna(count_str) or count_str == 'N/A' or count_str == '' or count_str is None:
        return 0
    
    if isinstance(count_str, (int, float)):
        return int(count_str)
    
    # Remove parentheses and extract number
    count_str = str(count_str).replace('(', '').replace(')', '').replace(',', '')
    
    # Extract number using regex
    count_match = re.search(r'(\d+)', count_str)
    if count_match:
        try:
            return int(count_match.group(1))
        except:
            return 0
    return 0

def parse_json_field(json_str):
    """Parse JSON string field"""
    if pd.isna(json_str) or json_str == 'N/A' or json_str == '' or json_str is None:
        return {}
    
    if isinstance(json_str, dict):
        return json_str
    
    if isinstance(json_str, list):
        return json_str
    
    try:
        return json.loads(json_str)
    except:
        return {}

def clean_text(text):
    """Clean text field"""
    if pd.isna(text) or text == 'N/A' or text is None:
        return ''
    
    text = str(text).strip()
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove HTML tags if any
    text = re.sub(r'<[^>]+>', '', text)
    return text

def extract_asin(url):
    """Extract ASIN from URL"""
    if pd.isna(url) or url is None:
        return np.nan
    
    asin_match = re.search(r'/dp/([A-Z0-9]{10})', str(url))
    if asin_match:
        return asin_match.group(1)
    
    # Try alternative pattern
    asin_match = re.search(r'/([A-Z0-9]{10})(?:\?|/|$)', str(url))
    return asin_match.group(1) if asin_match else np.nan

## Clean Products Data

In [18]:
def clean_products_data(df):
    """Clean products dataframe"""
    if df.empty:
        print("No data to clean!")
        return df
    
    df_clean = df.copy()
    
    print("Initial shape:", df_clean.shape)
    
    # 1. Handle duplicates
    print("\n1. Handling Duplicates...")
    if 'asin' in df_clean.columns:
        # Remove rows where ASIN is missing or N/A
        df_clean = df_clean[df_clean['asin'].notna() & (df_clean['asin'] != 'N/A')]
        
        duplicates_count = df_clean.duplicated(subset=['asin']).sum()
        print(f"   Duplicate ASINs: {duplicates_count}")
        df_clean = df_clean.drop_duplicates(subset=['asin'], keep='first')
    else:
        # If no ASIN column, use URL
        if 'url' in df_clean.columns:
            df_clean['asin'] = df_clean['url'].apply(extract_asin)
            df_clean = df_clean[df_clean['asin'].notna()]
            df_clean = df_clean.drop_duplicates(subset=['asin'], keep='first')
    
    print(f"   Shape after deduplication: {df_clean.shape}")
    
    # 2. Clean price columns
    print("\n2. Cleaning Prices...")
    price_columns = ['price', 'current_price', 'original_price']
    for col in price_columns:
        if col in df_clean.columns:
            df_clean[f'{col}_numeric'] = df_clean[col].apply(clean_price)
            valid_count = df_clean[f'{col}_numeric'].notna().sum()
            print(f"   {col}: {valid_count} valid values out of {len(df_clean)}")
    
    # 3. Clean rating columns
    print("\n3. Cleaning Ratings...")
    rating_columns = ['rating', 'detailed_rating']
    for col in rating_columns:
        if col in df_clean.columns:
            df_clean[f'{col}_numeric'] = df_clean[col].apply(clean_rating)
            valid_count = df_clean[f'{col}_numeric'].notna().sum()
            print(f"   {col}: {valid_count} valid values out of {len(df_clean)}")
    
    # 4. Clean reviews count
    print("\n4. Cleaning Reviews Count...")
    review_count_columns = ['reviews_count', 'detailed_reviews_count']
    for col in review_count_columns:
        if col in df_clean.columns:
            df_clean[f'{col}_numeric'] = df_clean[col].apply(clean_reviews_count)
    
    # 5. Clean text fields
    print("\n5. Cleaning Text Fields...")
    text_columns = ['title', 'full_title', 'description', 'features', 'brand', 'manufacturer']
    for col in text_columns:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].apply(clean_text)
    
    # 6. Handle missing values
    print("\n6. Handling Missing Values...")
    # Replace 'N/A' strings with NaN
    df_clean = df_clean.replace('N/A', np.nan)
    df_clean = df_clean.replace('', np.nan)
    
    # 7. Extract ASIN from URL if ASIN is missing
    print("\n7. Extracting ASINs...")
    if 'url' in df_clean.columns:
        mask = df_clean['asin'].isna() if 'asin' in df_clean.columns else pd.Series([True] * len(df_clean))
        if 'asin' not in df_clean.columns:
            df_clean['asin'] = df_clean['url'].apply(extract_asin)
        else:
            df_clean.loc[mask, 'asin'] = df_clean.loc[mask, 'url'].apply(extract_asin)
    
    # 8. Parse technical details
    print("\n8. Parsing Technical Details...")
    if 'technical_details' in df_clean.columns:
        df_clean['technical_details_parsed'] = df_clean['technical_details'].apply(parse_json_field)
    
    # 9. Parse reviews if stored as JSON array
    print("\n9. Parsing Reviews JSON...")
    if 'reviews' in df_clean.columns:
        df_clean['reviews_parsed'] = df_clean['reviews'].apply(parse_json_field)
        df_clean['reviews_count_actual'] = df_clean['reviews_parsed'].apply(
            lambda x: len(x) if isinstance(x, list) else 0
        )
    
    # 10. Create derived features
    print("\n10. Creating Derived Features...")
    # Discount percentage numeric
    if 'discount' in df_clean.columns:
        df_clean['discount_percentage'] = df_clean['discount'].apply(
            lambda x: clean_rating(str(x).replace('%', '')) if pd.notna(x) else np.nan
        )
    
    # Price difference
    if 'original_price_numeric' in df_clean.columns and 'current_price_numeric' in df_clean.columns:
        df_clean['price_difference'] = df_clean['original_price_numeric'] - df_clean['current_price_numeric']
    
    # 11. Convert boolean fields
    print("\n11. Converting Boolean Fields...")
    for col in ['is_prime', 'is_sponsored']:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].map({'Yes': True, 'No': False, 'yes': True, 'no': False, 
                                               True: True, False: False})
    
    # 12. Remove invalid rows
    print("\n12. Removing Invalid Rows...")
    # Remove rows with no title
    if 'title' in df_clean.columns:
        df_clean = df_clean[df_clean['title'].notna() & (df_clean['title'] != '')]
    
    print("\nFinal shape:", df_clean.shape)
    return df_clean

# Clean products data
if not products_df.empty:
    products_clean = clean_products_data(products_df)
else:
    products_clean = pd.DataFrame()
    print("No products data to clean!")

Initial shape: (621, 552)

1. Handling Duplicates...
   Duplicate ASINs: 0
   Shape after deduplication: (620, 552)

2. Cleaning Prices...
   price: 488 valid values out of 620
   current_price: 471 valid values out of 620
   original_price: 301 valid values out of 620

3. Cleaning Ratings...
   rating: 615 valid values out of 620
   detailed_rating: 610 valid values out of 620

4. Cleaning Reviews Count...

5. Cleaning Text Fields...

6. Handling Missing Values...

7. Extracting ASINs...

8. Parsing Technical Details...

9. Parsing Reviews JSON...

10. Creating Derived Features...

11. Converting Boolean Fields...

12. Removing Invalid Rows...

Final shape: (620, 563)


## Clean Reviews Data


In [19]:
def clean_reviews_data(df):
    """Clean reviews dataframe"""
    if df.empty:
        print("No reviews data to clean!")
        return df
    
    df_clean = df.copy()
    
    print("Initial shape:", df_clean.shape)
    
    # 1. Handle duplicates
    print("\n1. Handling Duplicates...")
    dedup_columns = []
    for col in ['product_url', 'reviewer_name', 'review_date', 'review_title']:
        if col in df_clean.columns:
            dedup_columns.append(col)
    
    if dedup_columns:
        df_clean = df_clean.drop_duplicates(subset=dedup_columns, keep='first')
        print(f"   Shape after deduplication: {df_clean.shape}")
    
    # 2. Clean rating
    print("\n2. Cleaning Ratings...")
    if 'review_rating' in df_clean.columns:
        df_clean['review_rating_numeric'] = df_clean['review_rating'].apply(clean_rating)
    
    # 3. Clean review date
    print("\n3. Cleaning Dates...")
    def parse_review_date(date_str):
        """Parse review date string to datetime"""
        if pd.isna(date_str) or date_str == 'N/A' or date_str is None:
            return pd.NaT
        
        date_str = str(date_str).strip()
        
        # Remove "Reviewed in" prefix
        date_str = re.sub(r'^Reviewed in\s+', '', date_str, flags=re.I)
        date_str = re.sub(r'^Reviewed in the\s+', '', date_str, flags=re.I)
        
        # Try different date formats
        date_formats = [
            '%d %B %Y',  # 15 January 2024
            '%B %d, %Y',  # January 15, 2024
            '%d %b %Y',  # 15 Jan 2024
            '%Y-%m-%d',  # 2024-01-15
        ]
        
        for fmt in date_formats:
            try:
                return pd.to_datetime(date_str, format=fmt)
            except:
                continue
        
        try:
            return pd.to_datetime(date_str)
        except:
            return pd.NaT
    
    if 'review_date' in df_clean.columns:
        df_clean['review_date_parsed'] = df_clean['review_date'].apply(parse_review_date)
    
    # 4. Clean text fields
    print("\n4. Cleaning Text Fields...")
    for col in ['review_title', 'review_body', 'reviewer_name']:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].apply(clean_text)
    
    # 5. Handle verified purchase
    print("\n5. Cleaning Verified Purchase...")
    if 'verified_purchase' in df_clean.columns:
        df_clean['verified_purchase'] = df_clean['verified_purchase'].map(
            {'Yes': True, 'No': False, 'yes': True, 'no': False, True: True, False: False}
        )
    
    # 6. Clean helpful votes
    print("\n6. Cleaning Helpful Votes...")
    def clean_helpful_votes(votes_str):
        """Extract number of helpful votes"""
        if pd.isna(votes_str) or votes_str == 'N/A' or votes_str is None:
            return 0
        
        votes_match = re.search(r'(\d+)', str(votes_str))
        if votes_match:
            return int(votes_match.group(1))
        return 0
    
    if 'helpful_votes' in df_clean.columns:
        df_clean['helpful_votes_numeric'] = df_clean['helpful_votes'].apply(clean_helpful_votes)
    
    # 7. Calculate review length
    print("\n7. Calculating Review Length...")
    if 'review_body' in df_clean.columns:
        df_clean['review_length'] = df_clean['review_body'].apply(
            lambda x: len(str(x).split()) if pd.notna(x) and x != '' else 0
        )
    
    # 8. Remove invalid reviews
    print("\n8. Removing Invalid Reviews...")
    # Remove reviews with no body and no title
    if 'review_body' in df_clean.columns and 'review_title' in df_clean.columns:
        df_clean = df_clean[
            ~((df_clean['review_body'].isna() | (df_clean['review_body'] == '')) & 
              (df_clean['review_title'].isna() | (df_clean['review_title'] == '')))
        ]
    
    # 9. Replace N/A with NaN
    df_clean = df_clean.replace('N/A', np.nan)
    
    print("\nFinal shape:", df_clean.shape)
    return df_clean

# Clean reviews data
if not reviews_df.empty:
    reviews_clean = clean_reviews_data(reviews_df)
else:
    reviews_clean = pd.DataFrame()
    print("No reviews data to clean!")

No reviews data to clean!


## Merge Products with Reviews (if separate files)

In [20]:
def merge_products_reviews(products_df, reviews_df):
    """Merge products with their reviews"""
    
    if products_df.empty:
        return products_df
    
    # Check if reviews are already in products (as JSON array)
    if 'reviews_parsed' in products_df.columns and products_df['reviews_parsed'].notna().any():
        print("Reviews already embedded in products data")
        return products_df
    
    if reviews_df.empty:
        print("No separate reviews data to merge")
        return products_df
    
    # Determine merge key
    merge_key = None
    if 'product_url' in reviews_df.columns and 'url' in products_df.columns:
        merge_key = 'product_url'
        left_key = 'url'
    elif 'product_id' in reviews_df.columns and 'id' in products_df.columns:
        merge_key = 'product_id'
        left_key = 'id'
    elif 'asin' in reviews_df.columns and 'asin' in products_df.columns:
        merge_key = 'asin'
        left_key = 'asin'
    
    if merge_key:
        # Aggregate reviews by product
        reviews_agg = reviews_df.groupby(merge_key).agg({
            'review_rating_numeric': ['mean', 'count', 'std'],
            'review_length': 'mean',
            'helpful_votes_numeric': 'sum',
        }).reset_index()
        
        # Flatten column names
        reviews_agg.columns = [merge_key, 'avg_review_rating', 'review_count', 
                               'rating_std', 'avg_review_length', 'total_helpful_votes']
        
        # Merge with products
        merged_df = products_df.merge(reviews_agg, left_on=left_key, right_on=merge_key, how='left')
        
        # Fill NaN values
        merged_df['avg_review_rating'] = merged_df['avg_review_rating'].fillna(0)
        merged_df['review_count'] = merged_df['review_count'].fillna(0).astype(int)
        merged_df['rating_std'] = merged_df['rating_std'].fillna(0)
        merged_df['avg_review_length'] = merged_df['avg_review_length'].fillna(0)
        merged_df['total_helpful_votes'] = merged_df['total_helpful_votes'].fillna(0).astype(int)
        
        print(f"Merged shape: {merged_df.shape}")
        return merged_df
    else:
        print("No common key found for merging")
        return products_df

# Merge data
if not products_clean.empty:
    merged_df = merge_products_reviews(products_clean, reviews_clean)
else:
    merged_df = pd.DataFrame()


Reviews already embedded in products data


### Additional Transformations

In [21]:
def additional_transformations(df):
    """Apply additional transformations"""
    if df.empty:
        return df
    
    df_transformed = df.copy()
    
    # 1. Create price category
    print("Creating price categories...")
    price_col = None
    if 'current_price_numeric' in df_transformed.columns:
        price_col = 'current_price_numeric'
    elif 'price_numeric' in df_transformed.columns:
        price_col = 'price_numeric'
    
    if price_col:
        bins = [0, 20, 50, 100, 200, 500, np.inf]
        labels = ['Under $20', '$20-$50', '$50-$100', '$100-$200', '$200-$500', 'Over $500']
        df_transformed['price_category'] = pd.cut(
            df_transformed[price_col], 
            bins=bins, 
            labels=labels,
            include_lowest=True
        )
    
    # 2. Create rating category
    print("Creating rating categories...")
    rating_col = None
    if 'rating_numeric' in df_transformed.columns:
        rating_col = 'rating_numeric'
    elif 'detailed_rating_numeric' in df_transformed.columns:
        rating_col = 'detailed_rating_numeric'
    elif 'avg_review_rating' in df_transformed.columns:
        rating_col = 'avg_review_rating'
    
    if rating_col:
        df_transformed['rating_category'] = pd.cut(
            df_transformed[rating_col],
            bins=[0, 2, 3, 4, 4.5, 5.1],
            labels=['Poor', 'Average', 'Good', 'Very Good', 'Excellent'],
            include_lowest=True
        )
    
    # 3. Create review count category
    print("Creating review count categories...")
    if 'review_count' in df_transformed.columns:
        df_transformed['review_count_category'] = pd.cut(
            df_transformed['review_count'],
            bins=[-1, 10, 100, 500, 1000, 5000, np.inf],
            labels=['0-10', '11-100', '101-500', '501-1000', '1001-5000', '5000+'],
            include_lowest=True
        )
    elif 'reviews_count_numeric' in df_transformed.columns:
        df_transformed['review_count_category'] = pd.cut(
            df_transformed['reviews_count_numeric'],
            bins=[-1, 10, 100, 500, 1000, 5000, np.inf],
            labels=['0-10', '11-100', '101-500', '501-1000', '1001-5000', '5000+'],
            include_lowest=True
        )
    
    # 4. Extract brand from technical details if missing
    print("Extracting brand from technical details...")
    if 'technical_details_parsed' in df_transformed.columns:
        def extract_brand(row):
            if pd.notna(row.get('brand')) and row['brand'] != '':
                return row['brand']
            tech_details = row.get('technical_details_parsed', {})
            if isinstance(tech_details, dict):
                return tech_details.get('Brand Name', tech_details.get('Manufacturer', ''))
            return ''
        
        df_transformed['brand_final'] = df_transformed.apply(extract_brand, axis=1)
    elif 'brand' in df_transformed.columns:
        df_transformed['brand_final'] = df_transformed['brand']
    
    # 5. Create has_video flag
    print("Creating video flag...")
    if 'video_url' in df_transformed.columns:
        df_transformed['has_video'] = df_transformed['video_url'].apply(
            lambda x: False if pd.isna(x) or x == 'N/A' or x == '' else True
        )
    
    # 6. Create has_discount flag
    print("Creating discount flag...")
    if 'discount_percentage' in df_transformed.columns:
        df_transformed['has_discount'] = df_transformed['discount_percentage'] > 0
    elif 'discount' in df_transformed.columns:
        df_transformed['has_discount'] = df_transformed['discount'].notna() & (df_transformed['discount'] != '')
    
    return df_transformed

# Apply transformations
if not merged_df.empty:
    merged_df = additional_transformations(merged_df)

Creating price categories...
Creating rating categories...
Creating review count categories...
Extracting brand from technical details...
Creating video flag...
Creating discount flag...


## Final Data Quality Checks

In [22]:
def data_quality_report(df):
    """Generate data quality report"""
    if df.empty:
        print("No data for quality check!")
        return pd.DataFrame()
    
    print("=" * 60)
    print("DATA QUALITY REPORT")
    print("=" * 60)
    
    print(f"\nTotal rows: {len(df)}")
    print(f"Total columns: {len(df.columns)}")
    
    # Missing values
    print("\nMissing Values Summary:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({
        'Missing Count': missing,
        'Missing %': missing_pct
    })
    missing_df = missing_df[missing_df['Missing Count'] > 0]
    if not missing_df.empty:
        print(missing_df)
    else:
        print("  No missing values!")
    
    # Duplicates
    print(f"\nDuplicate rows: {df.duplicated().sum()}")
    
    # Data types
    print("\nNumeric columns statistics:")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print(df[numeric_cols].describe())
    
    # Unique values in key columns
    print("\nUnique values in key columns:")
    for col in ['brand_final', 'price_category', 'rating_category', 'review_count_category']:
        if col in df.columns:
            print(f"  {col}: {df[col].nunique()} unique values")
    
    return missing_df

# Generate report
if not merged_df.empty:
    missing_report = data_quality_report(merged_df)

DATA QUALITY REPORT

Total rows: 620
Total columns: 569

Missing Values Summary:
                         Missing Count  Missing %
full_title                           5   0.806452
price                              132  21.290323
current_price                      149  24.032258
original_price                     319  51.451613
discount                           321  51.774194
...                                ...        ...
detailed_rating_numeric             10   1.612903
discount_percentage                321  51.774194
price_difference                   384  61.935484
price_category                     149  24.032258
rating_category                      5   0.806452

[548 rows x 2 columns]


TypeError: unhashable type: 'dict'

## Save Cleaned Data

In [ ]:
def save_cleaned_data(df, reviews_df, output_folder='cleaned_output'):
    """Save cleaned data to files"""
    if df.empty:
        print("No data to save!")
        return
    
    # Create output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Save to CSV
    products_csv = os.path.join(output_folder, f'cleaned_products_{timestamp}.csv')
    reviews_csv = os.path.join(output_folder, f'cleaned_reviews_{timestamp}.csv')
    
    df.to_csv(products_csv, index=False, encoding='utf-8')
    if not reviews_df.empty:
        reviews_df.to_csv(reviews_csv, index=False, encoding='utf-8')
    
    print(f"Saved cleaned products to: {products_csv}")
    if not reviews_df.empty:
        print(f"Saved cleaned reviews to: {reviews_csv}")
    
    # Save combined CSV
    combined_csv = os.path.join(output_folder, f'cleaned_combined_{timestamp}.csv')
    df.to_csv(combined_csv, index=False, encoding='utf-8')
    print(f"Saved combined data to: {combined_csv}")
    
    return products_csv, reviews_csv, combined_csv

# Save cleaned data
if not merged_df.empty:
    save_cleaned_data(merged_df, reviews_clean)


## Summary of Cleaning Steps

In [ ]:
if not products_df.empty:
    summary = {
        'Step': [
            'Initial products',
            'After removing duplicates',
            'After cleaning',
            'Final cleaned data'
        ],
        'Count': [
            len(products_df),
            len(products_clean) if not products_clean.empty else 0,
            len(merged_df) if not merged_df.empty else 0,
            len(merged_df) if not merged_df.empty else 0
        ]
    }
    
    summary_df = pd.DataFrame(summary)
    print(summary_df)
    
    print("\n" + "=" * 60)
    print("DATA CLEANING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    
    # Print final statistics
    if not merged_df.empty:
        print(f"\nFinal Statistics:")
        print(f"  Total Products: {len(merged_df)}")
        
        if 'brand_final' in merged_df.columns:
            print(f"  Unique Brands: {merged_df['brand_final'].nunique()}")
        
        if 'price_category' in merged_df.columns:
            print(f"  Price Categories: {merged_df['price_category'].nunique()}")
        
        if 'rating_category' in merged_df.columns:
            print(f"  Rating Categories: {merged_df['rating_category'].nunique()}")
        
        if 'has_video' in merged_df.columns:
            print(f"  Products with Video: {merged_df['has_video'].sum()}")
else:
    print("No data was loaded for cleaning!")

No data was loaded for cleaning!
